In [0]:
%run ../00_utils

# ***Silver Transform***

In [0]:
# Load du bronze
bronze_df = spark.table("nasa_meteor.bronze_full")

In [0]:
bronze_df.printSchema() # check schema

In [0]:
bronze_df.limit(10).display() # check to analyse the data 

In [0]:
silver_transform_approach = (
    bronze_df
    .withColumn("close_approach", explode("close_approach_data"))
)

In [0]:
# Rename all columns 
silver_rename = (
    silver_transform_approach
    .withColumn("close_approach", explode("close_approach_data"))
    .select(
        # ID
        col("id").alias("asteroid_id"),
        col("neo_reference_id"),
        col("designation"),
        col("name"),
        col("name_limited"),

        # General
        col("absolute_magnitude_h"),
        col("is_potentially_hazardous_asteroid"),
        col("is_sentry_object"),

        # URL
        col("nasa_jpl_url"),
        col("links.self").alias("api_url"),

        # Diameter KM
        col("estimated_diameter.kilometers.estimated_diameter_min").alias("diameter_min_km"),
        col("estimated_diameter.kilometers.estimated_diameter_max").alias("diameter_max_km"),

        # Diameter Meters
        col("estimated_diameter.meters.estimated_diameter_min").alias("diameter_min_m"),
        col("estimated_diameter.meters.estimated_diameter_max").alias("diameter_max_m"),

        # Diameter Miles
        col("estimated_diameter.miles.estimated_diameter_min").alias("diameter_min_miles"),
        col("estimated_diameter.miles.estimated_diameter_max").alias("diameter_max_miles"),

        # Diameter Feet
        col("estimated_diameter.feet.estimated_diameter_min").alias("diameter_min_feet"),
        col("estimated_diameter.feet.estimated_diameter_max").alias("diameter_max_feet"),

        # Orbit Data
        col("orbital_data.aphelion_distance"),
        col("orbital_data.ascending_node_longitude"),
        col("orbital_data.data_arc_in_days"),
        col("orbital_data.eccentricity"),
        col("orbital_data.epoch_osculation"),
        col("orbital_data.equinox"),
        col("orbital_data.first_observation_date"),
        col("orbital_data.last_observation_date"),
        col("orbital_data.inclination"),
        col("orbital_data.jupiter_tisserand_invariant"),
        col("orbital_data.mean_anomaly"),
        col("orbital_data.mean_motion"),
        col("orbital_data.minimum_orbit_intersection"),
        col("orbital_data.observations_used"),
        col("orbital_data.orbit_determination_date"),
        col("orbital_data.orbit_id"),
        col("orbital_data.orbit_uncertainty"),
        col("orbital_data.orbital_period"),
        col("orbital_data.perihelion_argument"),
        col("orbital_data.perihelion_distance"),
        col("orbital_data.perihelion_time"),
        col("orbital_data.semi_major_axis"),

        # Orbit Class
        col("orbital_data.orbit_class.orbit_class_type").alias("orbit_class_type"),
        col("orbital_data.orbit_class.orbit_class_description").alias("orbit_class_description"),
        col("orbital_data.orbit_class.orbit_class_range").alias("orbit_class_range"),

        # Close Approach
        col("close_approach.close_approach_date"),
        col("close_approach.close_approach_date_full"),
        col("close_approach.epoch_date_close_approach"),
        col("close_approach.orbiting_body"),

        # Miss Distance
        col("close_approach.miss_distance.astronomical").alias("miss_distance_au"),
        col("close_approach.miss_distance.kilometers").alias("miss_distance_km"),
        col("close_approach.miss_distance.miles").alias("miss_distance_miles"),
        col("close_approach.miss_distance.lunar").alias("miss_distance_lunar"),

        # Relative Velocity
        col("close_approach.relative_velocity.kilometers_per_hour").alias("velocity_kmh"),
        col("close_approach.relative_velocity.kilometers_per_second").alias("velocity_kms"),
        col("close_approach.relative_velocity.miles_per_hour").alias("velocity_mph"),

        # Metadata
        col("ingestion_timestamp"),
        col("source_file")
    )
)

In [0]:
# silver_rename.limit(50).display()
# silver_rename.where(col("asteroid_id") == 2001036).display()
# silver_rename.select('asteroid_id', 'velocity_kmh').distinct().sort(desc('velocity_kmh')).display()
# silver_rename.display()

In [0]:
# silver_rename.select([
#     count(when(col(c).isNull(), c)).alias(c)
#     for c in silver_rename.columns
# ]).display()

silver_null = (
    silver_rename
    .withColumn("name_limited", when(col("name_limited").isNull(), lit("Not Named")).otherwise(col("name_limited")))
)

In [0]:
silver_type = (
    silver_null
    .withColumn("asteroid_id", col("asteroid_id").cast(IntegerType()))
    .withColumn("neo_reference_id", col("neo_reference_id").cast(IntegerType()))
    .withColumn("designation", col("designation").cast(IntegerType()))
    .withColumn("aphelion_distance", col("aphelion_distance").cast(DoubleType()))
    .withColumn("ascending_node_longitude", col("ascending_node_longitude").cast(DoubleType()))
    .withColumn("eccentricity", col("eccentricity").cast(DoubleType()))
    .withColumn("epoch_osculation", col("epoch_osculation").cast(DoubleType()))
    .withColumn("first_observation_date", col("first_observation_date").cast(DateType()))
    .withColumn("last_observation_date", col("last_observation_date").cast(DateType()))
    .withColumn("inclination", col("inclination").cast(DoubleType()))
    .withColumn("jupiter_tisserand_invariant", col("jupiter_tisserand_invariant").cast(DoubleType()))
    .withColumn("mean_anomaly", col("mean_anomaly").cast(DoubleType()))
    .withColumn("mean_motion", col("mean_motion").cast(DoubleType()))
    .withColumn("minimum_orbit_intersection", col("minimum_orbit_intersection").cast(DoubleType()))
    .withColumn("orbit_determination_date", col("orbit_determination_date").cast(DateType()))
    .withColumn("orbit_id", col("orbit_id").cast(IntegerType()))
    .withColumn("orbit_uncertainty", col("orbit_uncertainty").cast(IntegerType()))
    .withColumn("orbital_period", col("orbital_period").cast(DoubleType()))
    .withColumn("perihelion_argument", col("perihelion_argument").cast(DoubleType()))
    .withColumn("perihelion_distance", col("perihelion_distance").cast(DoubleType()))
    .withColumn("perihelion_time", col("perihelion_time").cast(DoubleType()))
    .withColumn("semi_major_axis", col("semi_major_axis").cast(DoubleType()))
    .withColumn("close_approach_date", col("close_approach_date").cast(DateType()))
    .withColumn("close_approach_date_full", to_timestamp(col("close_approach_date_full"), "yyyy-MMM-dd HH:mm"))
    .withColumn("miss_distance_au", col("miss_distance_au").cast(DoubleType()))
    .withColumn("miss_distance_km", col("miss_distance_km").cast(DoubleType()))
    .withColumn("miss_distance_miles", col("miss_distance_miles").cast(DoubleType()))
    .withColumn("miss_distance_lunar", col("miss_distance_lunar").cast(DoubleType()))
    .withColumn("velocity_kmh", col("velocity_kmh").cast(DoubleType()))
    .withColumn("velocity_kms", col("velocity_kms").cast(DoubleType()))
    .withColumn("velocity_mph", col("velocity_mph").cast(DoubleType()))
)

# silver_type.select("velocity_kmh","miss_distance_km","eccentricity").summary().display()
# silver_type.display()

## _Create table_

### _Asteroid Table_

In [0]:
silver_asteroids = (
    silver_type
    .select("asteroid_id", "designation", "name", "name_limited", "absolute_magnitude_h", "is_potentially_hazardous_asteroid", "is_sentry_object", "diameter_min_km", "diameter_max_km", "diameter_min_m", "diameter_max_m", "diameter_min_miles", "diameter_max_miles", "diameter_min_feet", "diameter_max_feet", "ingestion_timestamp").dropDuplicates(["asteroid_id"])
)

# silver_asteroids.display()

### _Orbits Table_

In [0]:
silver_orbits = (
    silver_type
    .select("asteroid_id", "aphelion_distance", "ascending_node_longitude", "data_arc_in_days", "eccentricity", "epoch_osculation", "equinox", "first_observation_date", "last_observation_date", "inclination", "jupiter_tisserand_invariant", "mean_anomaly", "mean_motion", "minimum_orbit_intersection", "observations_used", "orbit_determination_date", "orbit_id", "orbit_uncertainty", "orbital_period", "perihelion_argument", "perihelion_distance", "perihelion_time", "semi_major_axis", "orbit_class_type", "orbit_class_description", "orbit_class_range", "ingestion_timestamp").dropDuplicates(["asteroid_id"])
)

# silver_orbits.display()

### _Close Approaches Table_

In [0]:
silver_close_approaches = (
    silver_type
    .select("asteroid_id", "close_approach_date", "close_approach_date_full", "epoch_date_close_approach", "orbiting_body", "miss_distance_au", "miss_distance_km", "miss_distance_miles", "miss_distance_lunar", "velocity_kmh", "velocity_kms", "velocity_mph", "ingestion_timestamp")
)

# silver_close_approaches.display()

## _Create table_

In [0]:
silver_asteroids.withColumn("silver_processed_at",current_timestamp()).write.mode("overwrite").format("delta").saveAsTable("nasa_meteor.silver_asteroids")
silver_orbits.withColumn("silver_processed_at",current_timestamp()).write.mode("overwrite").format("delta").saveAsTable("nasa_meteor.silver_orbits")
silver_close_approaches.withColumn("silver_processed_at",current_timestamp()).write.mode("overwrite").format("delta").saveAsTable("nasa_meteor.silver_close_approaches")